# ResNet Classifier - Make, Model, Year
Combining VMMRdb and StanfordCars into a single dataset to train on.

## Imports

In [131]:
import torchvision.transforms as tt
from torch.utils.data import Dataset
from torch.utils.data import ConcatDataset
from torch.utils.data import DataLoader, RandomSampler
from torch.utils.data import Subset
import pathlib
import os
import scipy.io as sio
from collections import Counter
import os
import re
from PIL import Image
import kagglehub
import random
from collections import defaultdict
from torch.utils.data import random_split
import torch

## StanfordCars Setup

### Download Dataset

In [5]:
import kagglehub

# Download latest version
path_stanford_cars = kagglehub.dataset_download("eduardo4jesus/stanford-cars-dataset")

print("Path to dataset files:", path_stanford_cars)

100%|██████████| 1.82G/1.82G [01:57<00:00, 16.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/eduardo4jesus/stanford-cars-dataset/versions/1


### StanfordCars Dataset Definition
Class Definition modified from PyTorch's implementation

In [35]:
import pathlib
import scipy.io as sio
from torchvision.datasets.folder import default_loader

class StanfordCars(Dataset):
    def __init__(self, root, global_class_to_idx, transform=None, max_images_per_class=None, class_normalizer=None):
        self.root = pathlib.Path(root)
        self.class_to_idx = global_class_to_idx
        self.loader = default_loader
        self.transform = transform
        self.class_normalizer = class_normalizer
        self.class_counts = Counter()

        devkit = self.root / "car_devkit/devkit"
        ann_path = devkit / "cars_train_annos.mat"
        img_dir = self.root / "cars_train/cars_train"

        annotations = sio.loadmat(ann_path, squeeze_me=True)["annotations"]
        raw_classes = sio.loadmat(devkit / "cars_meta.mat", squeeze_me=True)["class_names"].tolist()

        self.samples = []
        for ann in annotations:
            path = str(img_dir / ann["fname"])
            cls = raw_classes[ann["class"] - 1]

            if class_normalizer:
                cls = class_normalizer(cls)

            idx = self.class_to_idx.get(cls, None)
            if idx is None:
              continue

            if max_images_per_class and self.class_counts[cls] >= max_images_per_class:
              continue

            self.samples.append((path, idx))
            self.class_counts[cls] += 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = self.loader(path)
        if self.transform:
            img = self.transform(img)
        return img, y

## VMMRdb Setup

### Dataset Download

In [7]:
path_vmmrdb = kagglehub.dataset_download("prabashwara/vmmrdb-dataset")

print("Path to dataset files:", path_vmmrdb)

100%|██████████| 11.5G/11.5G [13:30<00:00, 15.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/prabashwara/vmmrdb-dataset/versions/1


### Custom VMMRDB Dataset Definition

In [36]:
class VMMRDB_Dataset(Dataset):
    def __init__(self, root_dir, global_class_to_idx, class_normalization, max_images_per_class=None, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = global_class_to_idx
        self.classes = []
        self.class_counts = Counter()

        # Images to class names
        for class_name in sorted(os.listdir(root_dir)):
          class_path = os.path.join(root_dir, class_name)
          if os.path.isdir(class_path):
            stripped = class_normalization(class_name)
            idx = self.class_to_idx.get(stripped, None)
            if idx is None:
              continue
            for img_name in os.listdir(class_path):
              if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                if max_images_per_class and self.class_counts[stripped] >= max_images_per_class:
                  continue

                self.image_paths.append(os.path.join(class_path, img_name))
                self.labels.append(idx)
                self.class_counts[stripped] += 1


        self.classes = self.class_to_idx.keys()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label

## Dataset Combination

### Normalization Function
This function is used to normalize the class labels in both the VMMRdb dataset and StanfordCars dataset

In [4]:
# Not part of model or make
EXTRA = {
    'sedan', 'coupe', 'convertible', 'hatchback', 'suv', 'wagon', 'van',
    'minivan', 'pickup', 'truck', 'cab', 'crew', 'extended', 'regular',
    'quad', 'club', 'cargo', 'passenger', 'sut', 'roadster', 'hybrid',
    'conv.', 'drophead', 'supercab', 'classic', 'superleggera',
    'abarth', 'activehybrid', 'series', 'electric',
}

# Normalizes classes in both datasets
def normalize_classes(car_class):
  car_class = re.sub('_', ' ', car_class).lower()
  car_class = re.sub("-", "", car_class)
  car_class_components = car_class.split(" ")
  car_class_components = [t for t in car_class_components if t not in EXTRA]
  car_class_components = ["mercedes benz" if t == "mercedesbenz" else t for t in car_class_components]
  car_class_components = ["rolls royce" if t == "rollsroyce" else t for t in car_class_components]
  return " ".join(car_class_components)

### Combining the two datasets
Using a minimum number of 40 images and a maximum number of 100 per class.

In [125]:
MIN_CLASSES = 40
MAX_CLASSES = 100

# Building global class_to_idx so we can combine the two datasets properly

def build_global_class_to_idx(stanford_root, vmmrdb_root, normalize_classes, min_classes):
    class_counts = Counter()


    # Stanford Cars
    devkit = pathlib.Path(stanford_root) / "car_devkit/devkit"
    ann_path = devkit / "cars_train_annos.mat"
    annotations = sio.loadmat(ann_path, squeeze_me=True)["annotations"]
    raw_classes = sio.loadmat(devkit / "cars_meta.mat", squeeze_me=True)["class_names"].tolist()

    # Normalizing class names + retrieving image count per class
    for ann in annotations:
        cls = normalize_classes(raw_classes[ann["class"] - 1])
        class_counts[cls] += 1

    # VMMRdb
    for class_name in sorted(os.listdir(vmmrdb_root)):
        class_path = os.path.join(vmmrdb_root, class_name)
        if os.path.isdir(class_path):

            # Normalizing class names + retrieving image count per class
            cls = normalize_classes(class_name)
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".jpeg", ".png")):
                    class_counts[cls] += 1

    # Removing classes with less than min_classes number of images
    trimmed_classes = {cls for cls, count in class_counts.items() if count >= min_classes}
    class_to_idx = {cls : idx for idx, cls in enumerate(sorted(trimmed_classes))}
    class_counts = {cls : count for cls,count in class_counts.items() if cls in trimmed_classes}

    return class_to_idx, class_counts

# Building the global class to idx and finding the number of images per class + removes classes with less than MIN_CLASSES images
global_class_to_idx, class_counts = build_global_class_to_idx(path_stanford_cars, path_vmmrdb, normalize_classes, MIN_CLASSES)

### Defining Training and Test Transforms

In [ ]:
# Defines image preprocessing for the training and test datasets
train_transform = tt.Compose([
    tt.Resize((224,224)),

    # Random Transformations
    tt.RandomHorizontalFlip(),
    tt.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    tt.RandomRotation(100),

    tt.ToTensor(),
    tt.Normalize([0.4367, 0.4331, 0.4279], [0.2685, 0.2652, 0.2675])
])

test_transform = tt.Compose([
    tt.Resize((224,224)),
    tt.ToTensor(),
    tt.Normalize([0.4367, 0.4331, 0.4279], [0.2685, 0.2652, 0.2675])
])

### Defining Datasets For Training and Testing

In [ ]:
# Need to create the dataset objects
# twice to properly create the test and training sets with the correct
# transforms tied to them

# Test dataset
stanford_test = StanfordCars(
    root=path_stanford_cars,
    global_class_to_idx=global_class_to_idx,
    class_normalizer=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=test_transform
)
vmmrdb_test = VMMRDB_Dataset(
    root_dir=path_vmmrdb,
    global_class_to_idx=global_class_to_idx,
    class_normalization=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=test_transform
)

# Train dataset
stanford_train = StanfordCars(
    root=path_stanford_cars,
    global_class_to_idx=global_class_to_idx,
    class_normalizer=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=train_transform
)
vmmrdb_train = VMMRDB_Dataset(
    root_dir=path_vmmrdb,
    global_class_to_idx=global_class_to_idx,
    class_normalization=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=train_transform
)

### Splitting the Training and Testing datasets

In [162]:
# Splitting into training and testing
train_full = ConcatDataset([stanford_train, vmmrdb_train])
test_full = ConcatDataset([stanford_test, vmmrdb_test])

# 80 - 20 split
train_size = int(0.8 * len(train_full))
test_size = len(train_full) - train_size

generator = torch.Generator().manual_seed(0)
indices = torch.randperm(len(train_full), generator=generator)
train_indices = indices[:train_size]
test_indices = indices[train_size:]

dataset_train = Subset(train_full, train_indices)
dataset_test = Subset(test_full, test_indices)

In [163]:
# batch size
batch_size = 512

train_loader = DataLoader(dataset_train, batch_size = batch_size, shuffle=True)
test_loader = DataLoader(dataset_test, batch_size = batch_size, shuffle=False)
